# Day 8 — Relationships, Migrations & FastAPI Integration

---

Yesterday we had one table (`users`). Real apps have **many tables that reference each other**: a user has posts, a post has tags, an order has line items.

Today:

1. Model **one-to-many** and **many-to-many** relationships in SQLAlchemy 2.0.
2. Understand **lazy vs eager loading** (and the dreaded N+1 query).
3. Use **cascades** to delete child rows automatically.
4. See how **Alembic migrations** evolve a schema over time.
5. Wire the whole thing into a **FastAPI** app with a clean Pydantic ↔ SQLAlchemy split.

In [ ]:
!pip install sqlalchemy fastapi uvicorn pydantic

## One-to-many — one `User`, many `Post`s

The shape:

```
users                posts
-----                -----
id    <-----+        id
name        +------- author_id   (FK -> users.id)
                     title
```

In SQLAlchemy you express this with a **`ForeignKey`** on the child and a **`relationship()`** on each side.

In [2]:
from __future__ import annotations
from sqlalchemy import create_engine, ForeignKey, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker

engine = create_engine("sqlite:///./day8_demo.db", echo=True)

class Base(DeclarativeBase):
    pass

class User(Base):
    __tablename__ = "users"
    id:    Mapped[int] = mapped_column(primary_key=True)
    name:  Mapped[str]
    email: Mapped[str] = mapped_column(unique=True)

    posts: Mapped[list["Post"]] = relationship(
        back_populates="author",
        cascade="all, delete-orphan",
    )

class Post(Base):
    __tablename__ = "posts"
    id:        Mapped[int] = mapped_column(primary_key=True)
    title:     Mapped[str]
    author_id: Mapped[int] = mapped_column(ForeignKey("users.id"))

    author: Mapped["User"] = relationship(back_populates="posts")

Base.metadata.drop_all(engine)   # fresh demo DB each run
Base.metadata.create_all(engine)  # create tables
SessionLocal = sessionmaker(bind=engine)

2026-07-23 23:30:40,065 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-23 23:30:40,066 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-07-23 23:30:40,066 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-07-23 23:30:40,067 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("users")
2026-07-23 23:30:40,067 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-07-23 23:30:40,067 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("posts")
2026-07-23 23:30:40,068 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-07-23 23:30:40,068 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("posts")
2026-07-23 23:30:40,069 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-07-23 23:30:40,069 INFO sqlalchemy.engine.Engine COMMIT
2026-07-23 23:30:40,069 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-23 23:30:40,070 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-07-23 23:30:40,070 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-07-23 23:30:40,070 INFO sql

In [3]:
with SessionLocal() as session:
    alice = User(name="Alice", email="a@x.com", posts=[
        Post(title="Hello"),
        Post(title="World"),
    ])
    session.add(alice)
    session.commit()

    # Walk the relationship in both directions
    a = session.execute(select(User).where(User.name == "Alice")).scalar_one()
    print("Alice's posts:", [p.title for p in a.posts])
    print("Post #1's author:", a.posts[0].author.name)

2026-07-23 23:30:48,103 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-23 23:30:48,105 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email) VALUES (?, ?)
2026-07-23 23:30:48,105 INFO sqlalchemy.engine.Engine [generated in 0.00039s] ('Alice', 'a@x.com')
2026-07-23 23:30:48,107 INFO sqlalchemy.engine.Engine INSERT INTO posts (title, author_id) VALUES (?, ?) RETURNING id
2026-07-23 23:30:48,108 INFO sqlalchemy.engine.Engine [generated in 0.00015s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('Hello', 1)
2026-07-23 23:30:48,109 INFO sqlalchemy.engine.Engine INSERT INTO posts (title, author_id) VALUES (?, ?) RETURNING id
2026-07-23 23:30:48,109 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('World', 1)
2026-07-23 23:30:48,110 INFO sqlalchemy.engine.Engine COMMIT
2026-07-23 23:30:48,111 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-23 23:30:48,114 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email 


## Many-to-many — `Post` ↔ `Tag`

A post can have many tags; a tag belongs to many posts. You need a third **association table** that just holds pairs of foreign keys.

In [5]:
from sqlalchemy import Table, Column

post_tags = Table(
    "post_tags",
    Base.metadata,
    Column("post_id", ForeignKey("posts.id"), primary_key=True),
    Column("tag_id", ForeignKey("tags.id"), primary_key=True),
)

class Tag(Base):
    __tablename__ = "tags"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(unique=True)

    posts: Mapped[list["Post"]] = relationship(
        secondary=post_tags, back_populates="tags"
    )

# Attach the other side of the m2m to Post
Post.tags = relationship("Tag", secondary=post_tags, back_populates="posts")

Base.metadata.create_all(engine)

with SessionLocal() as session:
    py = Tag(name="python")
    db = Tag(name="databases")
    p  = session.execute(select(Post).where(Post.title == "Hello")).scalar_one()
    p.tags = [py, db]
    session.commit()

    p = session.execute(select(Post).where(Post.title == "Hello")).scalar_one()
    print("Tags on 'Hello':", [t.name for t in p.tags])
    print("Posts tagged 'python':", [pp.title for pp in py.posts])

Tags on 'Hello': ['python', 'databases']
Posts tagged 'python': ['Hello']


## Lazy vs eager loading — and the N+1 problem

By default `relationship()` is **lazy**: SQLAlchemy fires a fresh `SELECT` the first time you touch `.posts` on each user.

Looks innocent, but in a loop it becomes a disaster:

```python
for user in session.execute(select(User)).scalars():   # 1 query
    print(user.name, len(user.posts))                  # +1 query PER USER
```

That's **N + 1** queries for N users. Fix with eager loading:

| Loader | What it does | When to use |
|---|---|---|
| `selectinload(User.posts)` | Second query with `WHERE post.author_id IN (...)` | One-to-many (most common) |
| `joinedload(User.posts)`   | Single query with `LEFT OUTER JOIN`              | One-to-one / small collections |

```python
from sqlalchemy.orm import selectinload
stmt = select(User).options(selectinload(User.posts))   # 2 queries total, regardless of N
```

Visual Comparison
Lazy Loading
Load Users
      │
      ▼

Users

 │
 ▼
Access posts?

 │
 ▼

SELECT posts
Repeated for every user.
User1 → Query
User2 → Query
User3 → Query
User4 → Query
...
selectinload()
Load Users
      │
      ▼

SELECT Users

      │
      ▼

SELECT Posts
WHERE author_id IN (...)

      │
      ▼

Everything Ready


## Cascades — deleting children automatically

When you delete a user, what should happen to their posts? You declare the policy on the **parent** side:

```python
posts: Mapped[list["Post"]] = relationship(
    back_populates="author",
    cascade="all, delete-orphan",
)
```

| Cascade option | Meaning |
|---|---|
| `save-update` | `add()` on parent stages children too (default) |
| `merge`       | `merge()` on parent merges children too (default) |
| `delete`      | Deleting parent deletes children |
| `delete-orphan` | Removing a child from the collection deletes it |
| `all`         | Shorthand for `save-update, merge, delete, ...` |
| `"all, delete-orphan"` | The usual "parent fully owns children" choice |

| Cascade          | What it does                                                               | Example                                         |
| ---------------- | -------------------------------------------------------------------------- | ----------------------------------------------- |
| `save-update`    | Save/update children when the parent is added or associated with a session | `session.add(user)` also persists `user.posts`  |
| `merge`          | Merge detached children when the parent is merged                          | `session.merge(user)` also merges `user.posts`  |
| `delete`         | Delete children when the parent is deleted                                 | `session.delete(user)` deletes the posts first  |
| `delete-orphan`  | Delete a child when it is removed from its parent and has no other parent  | `user.posts.remove(post)` deletes the post      |
| `refresh-expire` | Refresh/expire related objects together                                    | `session.refresh(user)` refreshes the posts too |
| `expunge`        | Remove related objects from the session                                    | `session.expunge(user)` also expunges the posts |
| `all`            | Includes `save-update`, `merge`, `refresh-expire`, `expunge`, and `delete` | Commonly used with `delete-orphan`              |


In [9]:
with SessionLocal() as session:
    a = session.execute(select(User).where(User.name == "Alice")).scalar_one()
    print("posts before:", session.execute(select(Post)).scalars().all())
    session.delete(a)
    session.commit()
    print("posts after :", session.execute(select(Post)).scalars().all())  # gone

2026-07-23 23:40:01,180 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-07-23 23:40:01,181 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email 
FROM users 
WHERE users.name = ?
2026-07-23 23:40:01,181 INFO sqlalchemy.engine.Engine [cached since 553.1s ago] ('Alice',)
2026-07-23 23:40:01,183 INFO sqlalchemy.engine.Engine SELECT posts.id, posts.title, posts.author_id 
FROM posts
2026-07-23 23:40:01,183 INFO sqlalchemy.engine.Engine [generated in 0.00035s] ()
posts before: [<__main__.Post object at 0x10e612cd0>, <__main__.Post object at 0x10e612d50>]
2026-07-23 23:40:01,184 INFO sqlalchemy.engine.Engine SELECT posts.id AS posts_id, posts.title AS posts_title, posts.author_id AS posts_author_id 
FROM posts 
WHERE ? = posts.author_id
2026-07-23 23:40:01,184 INFO sqlalchemy.engine.Engine [cached since 553.1s ago] (1,)
2026-07-23 23:40:01,188 INFO sqlalchemy.engine.Engine DELETE FROM posts WHERE posts.id = ?
2026-07-23 23:40:01,188 INFO sqlalchemy.engine.Engine [generat

## Alembic — schema migrations

`Base.metadata.create_all()` is fine for **new** tables in development. But the moment you ship and need to *change* a table (add a column, rename, drop), you need **migrations**.

[Alembic](https://alembic.sqlalchemy.org/) is the official SQLAlchemy tool for this. Typical workflow:

```bash
pip install alembic
alembic init alembic                                  # one-time scaffolding

# Point alembic at your metadata in alembic/env.py:
#     from myapp.models import Base
#     target_metadata = Base.metadata

alembic revision --autogenerate -m "add posts table"  # creates a versioned migration file
alembic upgrade head                                  # apply it
alembic downgrade -1                                  # roll back one step
```

Each migration is a Python file under `alembic/versions/` with an `upgrade()` and `downgrade()`. Commit them to git like any other source file. We won't run Alembic in this notebook — just know the shape of the workflow.

## Integrating with FastAPI — the `get_db` dependency

A FastAPI endpoint needs a session, but it must not leak one. The pattern is a **dependency that yields**:

```python
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
```

Then any endpoint can declare `db: Session = Depends(get_db)` — FastAPI opens a session before the call and closes it after, even on errors.

## Pydantic vs SQLAlchemy — separation of concerns

This is the single most important habit to build today.

| | SQLAlchemy model | Pydantic schema |
|---|---|---|
| **Job** | Maps to a DB row | Validates an HTTP body / shapes a response |
| **Lives in** | `models.py` | `schemas.py` |
| **Inherits** | `Base(DeclarativeBase)` | `BaseModel` |
| **Fields** | `Mapped[...]` | typed annotations + `Field(...)` |
| **Has secrets?** | Yes (`hashed_password`, internal flags) | No — only what's safe to expose |

The two **must not be the same class**. Keep them separate, then translate at the edges:

```python
class UserOut(BaseModel):
    id: int
    name: str
    email: str
    model_config = ConfigDict(from_attributes=True)   # reads SQLAlchemy attributes directly
```

With `from_attributes=True`, `UserOut.model_validate(db_user)` works — Pydantic reads attributes instead of dict keys.

## End-to-end demo: FastAPI + SQLAlchemy + Pydantic

Let's stand up a tiny app **in this notebook** and hit it with `TestClient` — no separate server process needed.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, EmailStr
from sqlalchemy.orm import Session

# Fresh tiny DB for the demo
demo_engine = create_engine(
    "sqlite:///./day8_api_demo.db",
    connect_args={"check_same_thread": False},
)
Base.metadata.drop_all(demo_engine)
Base.metadata.create_all(demo_engine)
DemoSession = sessionmaker(bind=demo_engine)

def get_db():
    db = DemoSession()
    try:
        yield db
    finally:
        db.close()

# --- Pydantic schemas (API contract) ---
class UserCreate(BaseModel):
    name: str
    email: str

class UserOut(BaseModel):
    id: int
    name: str
    email: str
    model_config = ConfigDict(from_attributes=True)

app = FastAPI()

@app.post("/users", response_model=UserOut)
def create_user(payload: UserCreate, db: Session = Depends(get_db)):
    user = User(name=payload.name, email=payload.email)
    db.add(user); db.commit(); db.refresh(user)
    return user        # Pydantic reads attrs because from_attributes=True

@app.get("/users/{user_id}", response_model=UserOut)
def get_user(user_id: int, db: Session = Depends(get_db)):
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(404, "user not found")
    return user

In [ ]:
client = TestClient(app)

r = client.post("/users", json={"name": "Alice", "email": "alice@x.com"})
print("POST /users  ->", r.status_code, r.json())

uid = r.json()["id"]
r = client.get(f"/users/{uid}")
print(f"GET /users/{uid} ->", r.status_code, r.json())

r = client.get("/users/9999")
print("GET /users/9999 ->", r.status_code, r.json())

### Why this layout pays off

- Add a column to `User` (e.g. `hashed_password`) — `UserOut` doesn't expose it. No accidental leak.
- Validate the request (`UserCreate`) before it ever touches the DB — bad payloads → 422, not a corrupt row.
- Swap the DB later (Postgres, MySQL) — only the SQLAlchemy side changes.

## Recap

- **One-to-many**: `ForeignKey` on the child + `relationship(back_populates=...)` on both sides.
- **Many-to-many**: an association `Table` + `relationship(secondary=...)`.
- Default loading is **lazy** → N+1. Use `selectinload` / `joinedload` to fix.
- **Cascade `all, delete-orphan`** = parent owns its children; deleting the parent deletes them.
- **Alembic** evolves your schema across deployments — never edit a shipped table by hand.
- In FastAPI, use a `get_db` yield-dependency for per-request sessions.
- Always keep **SQLAlchemy models** (DB row) and **Pydantic schemas** (API contract) as separate classes — bridge them with `ConfigDict(from_attributes=True)`.